#  Preprocesamiento de Texto para Detección de Toxicidad

**Autor:** Gema
**Fecha:** 2026  
**Objetivo:** Limpiar y preparar el texto del dataset para entrenamiento del modelo

---

##  Qué vamos a hacer en este notebook

1. ✅ Cargar el dataset original
2. ✅ Limpiar el texto (minúsculas, URLs, caracteres especiales)
3. ✅ Eliminar stopwords en inglés
4. ✅ Aplicar stemming o lematización
5. ✅ Guardar dataset limpio con la columna `text_procesado`

---

## 📊 Dataset

- **Entrada:** `data/raw/youtoxic_orginal.csv`
- **Salida:** `data/processed/youtoxic_comments_cleaned.csv`
- **Target:** IsToxic (binario: True/False)
- **Balance:** 53.8% Normal vs 46.2% Tóxico

---

## 🛠️ Técnicas de Preprocesamiento

1. **Convertir a minúsculas:** "HELLO" → "hello"
2. **Eliminar URLs:** "Check https://example.com" → "Check"
3. **Eliminar menciones:** "@user text" → "text"
4. **Eliminar números:** "I hate 123" → "I hate"
5. **Eliminar caracteres especiales:** "What!!!" → "What"
6. **Eliminar stopwords:** "this is a test" → "test"
7. **Stemming:** "running runs ran" → "run run run"

---

**Comenzamos →**

In [3]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import re
import warnings

# Librerías de NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer

# Configuración
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 200)

# Descargar recursos de NLTK (solo primera vez)
print(" Descargando recursos de NLTK...")
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('omw-1.4', quiet=True)

print(" Librerías importadas correctamente")
print(f" Stopwords en inglés: {len(stopwords.words('english'))} palabras")

 Descargando recursos de NLTK...
 Librerías importadas correctamente
 Stopwords en inglés: 198 palabras


In [6]:
# Cargar el dataset original
print("="*80)
print("📂 CARGANDO DATASET")
print("="*80)

# Cargar
df = pd.read_csv('../data/raw/youtoxic_original.csv')

print(f" Dataset cargado correctamente")
print(f"\n Dimensiones: {df.shape[0]:,} comentarios × {df.shape[1]} columnas")
print(f"\n Columnas disponibles:")
print(df.columns.tolist())

# Mostrar primeras 3 filas
print(f"\n{'='*80}")
print(" PRIMERAS 3 FILAS")
print(f"{'='*80}")
print(df[['Text', 'IsToxic']].head(5))

📂 CARGANDO DATASET
 Dataset cargado correctamente

 Dimensiones: 1,000 comentarios × 15 columnas

 Columnas disponibles:
['CommentId', 'VideoId', 'Text', 'IsToxic', 'IsAbusive', 'IsThreat', 'IsProvocative', 'IsObscene', 'IsHatespeech', 'IsRacist', 'IsNationalist', 'IsSexist', 'IsHomophobic', 'IsReligiousHate', 'IsRadicalism']

 PRIMERAS 3 FILAS
                                                                                                                                                                                                      Text  \
0  If only people would just take a step back and not make this case about them, because it wasn't about anyone except the two people in that situation.  To lump yourself into this mess and take matt...   
1                                                               Law enforcement is not trained to shoot to apprehend.  They are trained to shoot to kill.  And I thank Wilson for killing that punk bitch.   
2  \r\nDont you reckon them 'black 

In [7]:
# Seleccionar solo las columnas necesarias
print("="*80)
print(" SELECCIONANDO COLUMNAS NECESARIAS")
print("="*80)

# Columnas que vamos a mantener
columnas_necesarias = ['CommentId', 'VideoId', 'Text', 'IsToxic']

# Filtrar DataFrame
df = df[columnas_necesarias]

print(f" Columnas seleccionadas: {df.columns.tolist()}")
print(f"\n Nuevo tamaño: {df.shape[0]:,} filas × {df.shape[1]} columnas")

# Verificar balance de IsToxic
print(f"\n⚖️ BALANCE DEL TARGET:")
print(df['IsToxic'].value_counts())
print(f"\nPorcentajes:")
print(df['IsToxic'].value_counts(normalize=True) * 100)

 SELECCIONANDO COLUMNAS NECESARIAS
 Columnas seleccionadas: ['CommentId', 'VideoId', 'Text', 'IsToxic']

 Nuevo tamaño: 1,000 filas × 4 columnas

⚖️ BALANCE DEL TARGET:
IsToxic
False    538
True     462
Name: count, dtype: int64

Porcentajes:
IsToxic
False    53.8
True     46.2
Name: proportion, dtype: float64
